# EYEWAZ — train your Urdu Piper voice

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Upload your
`dataset-<VOICE>.zip` via the Files panel (📁). Then Runtime → **Run all**.

No kernel restart needed — we run Piper under a Python 3.10 it needs, while the
notebook kernel stays as-is. So your uploaded data is NOT wiped.


### 1. Check the GPU is on


In [ ]:
!nvidia-smi -L || print('NO GPU — set Runtime > Change runtime type > GPU first!')


### 2. Make a Python 3.10 (Piper's phonemizer has no wheels for newer Python)
Uses `uv` to fetch a standalone CPython 3.10 and puts it first on PATH, so every
`!python`/`!pip` below is 3.10. No restart.


In [ ]:
!pip install -q uv
!uv venv -p 3.10 --seed /content/p310
import os
os.environ['PATH'] = '/content/p310/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/content/p310'
!python --version   # -> Python 3.10.x (subprocess; the kernel itself stays newer, that's fine)


### 3. Install Piper training (+ the cp310 phonemizer wheel)


In [ ]:
WHL = 'https://github.com/rhasspy/piper-phonemize/releases/download/v1.1.0/piper_phonemize-1.1.0-cp310-cp310-manylinux_2_28_x86_64.whl'
!pip install -q {WHL}
import os
if not os.path.exists('/content/piper'):
    !git clone -q https://github.com/rhasspy/piper /content/piper
%cd /content/piper/src/python
!pip install -q -e .
!pip install -q -r requirements.txt
!bash build_monotonic_align.sh
print('Piper training installed ✓')


### 4. Settings + unzip your dataset
An upload button appears if the zip isn't already in /content.


In [ ]:
import os
VOICE = 'male'          # must match your uploaded dataset-<VOICE>.zip
DATA  = f'/content/dataset-{VOICE}'
OUT   = f'/content/train-{VOICE}'
zip_path = f'/content/dataset-{VOICE}.zip'
if not os.path.exists(zip_path):
    from google.colab import files
    print(f'Select dataset-{VOICE}.zip to upload:')
    up = files.upload()
    for name in up:
        if name != os.path.basename(zip_path): os.replace(f'/content/{name}', zip_path)
assert os.path.exists(zip_path), f'dataset-{VOICE}.zip not uploaded'
!unzip -oq {zip_path} -d /content/
print('clips:', len(os.listdir(f'{DATA}/wav')))


### 5. Preprocess (Urdu phonemes via espeak-ng)


In [ ]:
!python -m piper_train.preprocess \
  --language ur \
  --input-dir {DATA} \
  --output-dir {OUT} \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050


### 6. Pretrained checkpoint to fine-tune from
Lessac (en_US) medium — acoustic model transfers; Urdu phonemes from espeak-ng.


In [ ]:
CKPT='https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!wget -q --show-progress -O /content/pretrained.ckpt "$CKPT"
import os; print('checkpoint MB:', round(os.path.getsize('/content/pretrained.ckpt')/1e6,1))


### 7. Train
Watch the audio samples under the ▶ outputs. ~6 min of data = a **pilot** (rough).
**Stop the cell** (■) when samples sound good, then run the next cell.


In [ ]:
!python -m piper_train \
  --dataset-dir {OUT} \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --max_epochs 4000 \
  --checkpoint-epochs 5 \
  --quality medium \
  --resume_from_checkpoint /content/pretrained.ckpt


### 8. Export to ONNX + download


In [ ]:
ck = !ls -t {OUT}/lightning_logs/version_0/checkpoints/*.ckpt | head -1
ck = ck[0]; print('exporting', ck)
!python -m piper_train.export_onnx {ck} /content/eyewaz-urdu-{VOICE}.onnx
!cp {OUT}/config.json /content/eyewaz-urdu-{VOICE}.onnx.json
from google.colab import files
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx')
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx.json')


### Done
Send me `eyewaz-urdu-<VOICE>.onnx` + `.onnx.json` and I'll wire the voice into every EYEWAZ surface.
